# GeoAI Land Cover Classifier — 03 · Entrenamiento y comparación

**Dataset:** EuroSAT RGB (Sentinel-2) · **Entorno:** Google Colab · **Estado:** experimentos exploratorios.

**Ejecución:** `Entorno de ejecución → Ejecutar todo` desde una sesión nueva. Cada notebook prepara las variables que necesita; no depende de otro notebook abierto.

**Datos:** `./data` se descarga temporalmente en Colab. **Nunca utilices el conjunto test para escoger modelos.**

**Reproducibilidad:** `random_state=42`; pueden existir diferencias de tiempo o pequeñas variaciones entre versiones de dependencias.


## Preparación reproducible

Se reconstruyen las particiones y características a partir del dataset público, porque los notebooks de Colab **no comparten memoria**. Esto repite el *preprocesamiento*, no supone depender de ejecutar el notebook 02.

In [ ]:

import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import PIL

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Pillow:", PIL.__version__)


In [ ]:

from torchvision.datasets import EuroSAT

dataset = EuroSAT(
    root="./data",
    download=True
)

print("Número de imágenes:", len(dataset))
print("Categorías:", dataset.classes)

imagen, etiqueta = dataset[0]

print("Tamaño de la imagen:", imagen.size)
print("Primera categoría:", dataset.classes[etiqueta])


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 04: División estratificada del dataset
# ============================================================

# train_test_split divide una colección de datos en
# subconjuntos aleatorios, de forma reproducible.
from sklearn.model_selection import train_test_split

# NumPy nos permite trabajar con arrays numéricos.
import numpy as np

# ------------------------------------------------------------
# 1. Preparar índices y etiquetas
# ------------------------------------------------------------

# Cada índice identifica una imagen del dataset.
# Por ejemplo, el índice 0 corresponde a dataset[0].
#
# No cargamos todas las imágenes en memoria:
# trabajamos únicamente con sus posiciones.
indices = np.arange(len(dataset))

# Convertimos las etiquetas a un array de NumPy.
etiquetas = np.array(dataset.targets)

# ------------------------------------------------------------
# 2. Primera división: Train (70 %) y temporal (30 %)
# ------------------------------------------------------------

# test_size=0.30 reserva el 30 % para dividirlo después
# entre validación y prueba.
#
# random_state=42 fija la semilla aleatoria y permite
# reproducir exactamente esta división.
#
# stratify=etiquetas conserva las proporciones de clases.
indices_train, indices_temporales = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=etiquetas
)

# ------------------------------------------------------------
# 3. Segunda división: Validation (15 %) y Test (15 %)
# ------------------------------------------------------------

# Obtenemos las etiquetas de las imágenes temporales.
etiquetas_temporales = etiquetas[indices_temporales]

# Dividimos el 30 % reservado en dos mitades iguales.
#
# El 50 % del 30 % original equivale al 15 % del total.
indices_val, indices_test = train_test_split(
    indices_temporales,
    test_size=0.50,
    random_state=42,
    stratify=etiquetas_temporales
)

# ------------------------------------------------------------
# 4. Mostrar los tamaños
# ------------------------------------------------------------

print("Entrenamiento:", len(indices_train))
print("Validación:", len(indices_val))
print("Prueba:", len(indices_test))

# Verificamos que no falte ninguna imagen.
assert (
    len(indices_train)
    + len(indices_val)
    + len(indices_test)
    == len(dataset)
), "Error: el número de imágenes no coincide."

# Comprobamos que ningún índice esté repetido
# entre los tres conjuntos.
assert len(
    set(indices_train)
    | set(indices_val)
    | set(indices_test)
) == len(dataset), "Error: hay índices duplicados."

print("División completada correctamente.")


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 07: Preparación de una muestra de entrenamiento
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Seleccionar 2.000 imágenes de entrenamiento
# ------------------------------------------------------------

# Partimos exclusivamente de indices_train.
# De esta forma no incorporamos imágenes de validación
# ni de prueba durante el entrenamiento.
#
# train_size=2000 selecciona exactamente 2.000 imágenes.
#
# random_state=42 permite reproducir la selección.
#
# stratify mantiene aproximadamente las proporciones
# originales de las diez categorías.

indices_muestra, _ = train_test_split(
    indices_train,
    train_size=2000,
    random_state=42,
    stratify=etiquetas[indices_train]
)

print("Imágenes seleccionadas:", len(indices_muestra))

# ------------------------------------------------------------
# 2. Crear una función de preparación de imágenes
# ------------------------------------------------------------

def preparar_imagen(indice):
    """
    Convierte una imagen de EuroSAT en un vector numérico.

    Parámetros
    ----------
    indice : int
        Posición de la imagen dentro del dataset.

    Devuelve
    --------
    vector : np.ndarray
        Vector de 12.288 características normalizadas.

    etiqueta : int
        Identificador numérico de la categoría.
    """

    # Recuperamos una imagen RGB y su etiqueta.
    imagen, etiqueta = dataset[int(indice)]

    # Convertimos la imagen a un array de NumPy.
    imagen_array = np.asarray(imagen)

    # Normalizamos los valores RGB a [0, 1].
    # float32 reduce el consumo de memoria frente a float64.
    imagen_normalizada = (
        imagen_array.astype(np.float32) / 255.0
    )

    # Convertimos la matriz (64, 64, 3)
    # en un vector de 12.288 características.
    vector = imagen_normalizada.flatten()

    return vector, etiqueta


# ------------------------------------------------------------
# 3. Procesar todas las imágenes de la muestra
# ------------------------------------------------------------

# Creamos dos listas:
# una para los vectores y otra para las etiquetas.
caracteristicas = []
etiquetas_muestra = []

for indice in indices_muestra:

    vector, etiqueta = preparar_imagen(indice)

    caracteristicas.append(vector)
    etiquetas_muestra.append(etiqueta)

# ------------------------------------------------------------
# 4. Convertir las listas en arrays
# ------------------------------------------------------------

# Cada fila de X representa una imagen.
# Cada columna representa una característica RGB.
X_train_sample = np.stack(caracteristicas)

# y contiene las etiquetas numéricas correspondientes.
y_train_sample = np.array(etiquetas_muestra)

# ------------------------------------------------------------
# 5. Comprobar las dimensiones
# ------------------------------------------------------------

print("Forma de X:", X_train_sample.shape)
print("Forma de y:", y_train_sample.shape)

print("Tipo de dato X:", X_train_sample.dtype)

print("Valor mínimo:", X_train_sample.min())
print("Valor máximo:", X_train_sample.max())

# Verificamos que haya una etiqueta por imagen.
assert len(X_train_sample) == len(y_train_sample)

# Verificamos que las imágenes tengan 12.288 características.
assert X_train_sample.shape[1] == 64 * 64 * 3

print("Muestra preparada correctamente.")



## 08. Primer modelo: clasificación logística con SGD

### Objetivo

Entrenar nuestro primer clasificador de imágenes
satelitales usando 2.000 imágenes de EuroSAT RGB.

### Conceptos importantes

**Modelo:** algoritmo que aprende relaciones entre las
características de entrada (X) y las etiquetas (y).

**Entrenamiento:** proceso de ajustar los parámetros
del modelo utilizando ejemplos etiquetados.

**Clasificador lineal:** modelo que combina las
características mediante pesos para producir predicciones.

**Descenso de gradiente estocástico (SGD):**
método de optimización que actualiza los parámetros
a partir de ejemplos de entrenamiento.

**Baseline:** primer modelo de referencia contra el que
compararemos técnicas posteriores.

### Decisión técnica

Utilizamos SGDClassifier con pérdida logística porque:

- Es adecuado para experimentar con muchas características.
- Permite entrenar modelos lineales de forma eficiente.
- Proporciona una referencia sencilla.
- Nos permite estudiar el funcionamiento del entrenamiento
  antes de introducir modelos más complejos.

### Limitaciones

El modelo recibe píxeles RGB aplanados y no utiliza
explícitamente la estructura espacial de la imagen.

No esperamos necesariamente un rendimiento alto.
El objetivo inicial es establecer una referencia medible.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 08: Entrenamiento del primer modelo (baseline)
# ============================================================

import time

# SGDClassifier permite entrenar clasificadores lineales
# mediante descenso de gradiente estocástico.
from sklearn.linear_model import SGDClassifier

# accuracy_score mide el porcentaje de predicciones
# correctas sobre el total de ejemplos evaluados.
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Definir nuestro primer modelo
# ------------------------------------------------------------

modelo_baseline = SGDClassifier(
    loss="log_loss",      # Utiliza una función de pérdida logística.
    max_iter=1000,       # Número máximo de épocas de entrenamiento.
    tol=1e-3,            # Criterio de parada por mejora insuficiente.
    random_state=42,     # Permite reproducir la aleatoriedad.
    n_jobs=-1            # Utiliza los núcleos disponibles cuando aplica.
)

# ------------------------------------------------------------
# 2. Entrenar el modelo
# ------------------------------------------------------------

# Registramos el instante inicial para medir el tiempo.
inicio = time.time()

# fit() aprende los parámetros del modelo.
#
# X_train_sample: los valores RGB de nuestras imágenes.
# y_train_sample: la clase real de cada imagen.
#
# IMPORTANTE:
# Entrenamos exclusivamente con los datos de entrenamiento.
# No utilizamos validación ni prueba en este paso.
modelo_baseline.fit(
    X_train_sample,
    y_train_sample
)

# Calculamos el tiempo transcurrido.
tiempo_entrenamiento = time.time() - inicio

# ------------------------------------------------------------
# 3. Obtener predicciones sobre entrenamiento
# ------------------------------------------------------------

# predict() utiliza el modelo entrenado para estimar
# la categoría de cada imagen.
predicciones_train = modelo_baseline.predict(
    X_train_sample
)

# ------------------------------------------------------------
# 4. Calcular la accuracy de entrenamiento
# ------------------------------------------------------------

# Comparamos las predicciones con las etiquetas reales.
accuracy_train = accuracy_score(
    y_train_sample,
    predicciones_train
)

# ------------------------------------------------------------
# 5. Mostrar los resultados
# ------------------------------------------------------------

print("Modelo:", type(modelo_baseline).__name__)

print(
    "Tiempo de entrenamiento:",
    round(tiempo_entrenamiento, 2),
    "segundos"
)

print(
    "Accuracy de entrenamiento:",
    round(accuracy_train * 100, 2),
    "%"
)

print(
    "Número de clases aprendidas:",
    len(modelo_baseline.classes_)
)

print("Entrenamiento completado.")



## 09. Evaluación inicial del baseline

### Objetivo

Evaluar nuestro primer modelo de clasificación sobre
1.000 imágenes del conjunto de validación.

Estas imágenes no han sido utilizadas durante
el entrenamiento.

### Conceptos

**Generalización:**
Capacidad del modelo para realizar predicciones
sobre ejemplos que no ha visto durante el entrenamiento.

**Accuracy de validación:**
Porcentaje de predicciones correctas en un conjunto
independiente del utilizado para aprender.

**Overfitting (sobreajuste):**
Situación en la que un modelo aprende demasiado
las particularidades del entrenamiento y funciona
peor con datos nuevos.

**Macro-F1:**
Media de los valores F1 de todas las clases,
dando el mismo peso a cada categoría.

### Decisiones técnicas

- Usaremos una muestra estratificada de 1.000 imágenes.
- Mantendremos exactamente el mismo preprocesamiento RGB.
- No volveremos a entrenar el modelo.
- No utilizaremos el conjunto de prueba.
- Compararemos las métricas de entrenamiento y validación.

### Limitación

Esta evaluación utiliza una división aleatoria por
imagen y no garantiza generalización a regiones
geográficas independientes.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 09: Evaluación inicial del modelo
# ============================================================

import time
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Seleccionar una muestra del conjunto de validación
# ------------------------------------------------------------

# indices_val contiene las 4.050 imágenes reservadas
# anteriormente para validar nuestros modelos.
#
# Elegimos 1.000 de forma estratificada.
# Esto conserva aproximadamente la distribución
# de las diez categorías.
indices_val_muestra, _ = train_test_split(
    indices_val,
    train_size=1000,
    random_state=42,
    stratify=etiquetas[indices_val]
)

# ------------------------------------------------------------
# 2. Preparar las imágenes de validación
# ------------------------------------------------------------

# Reutilizamos preparar_imagen(), la misma función
# utilizada para crear los datos de entrenamiento.
#
# Es fundamental aplicar el mismo preprocesamiento:
# RGB -> float32 -> normalización [0, 1] -> flatten.
#
# No calculamos parámetros estadísticos nuevos
# ni ajustamos el modelo con estos datos.

X_val_lista = []
y_val_lista = []

for indice in indices_val_muestra:

    vector, etiqueta = preparar_imagen(indice)

    X_val_lista.append(vector)
    y_val_lista.append(etiqueta)

# Convertimos las listas en arrays de NumPy.
X_val_sample = np.stack(X_val_lista)
y_val_sample = np.array(y_val_lista)

# ------------------------------------------------------------
# 3. Realizar predicciones
# ------------------------------------------------------------

# predict() utiliza el modelo que ya entrenamos.
# No modificamos sus parámetros.
inicio = time.time()

predicciones_val = modelo_baseline.predict(
    X_val_sample
)

tiempo_inferencia = time.time() - inicio

# ------------------------------------------------------------
# 4. Calcular las métricas
# ------------------------------------------------------------

# Accuracy: proporción total de aciertos.
accuracy_val = accuracy_score(
    y_val_sample,
    predicciones_val
)

# Macro-F1: calcula F1 para cada una de las diez clases
# y obtiene la media sin favorecer las clases mayoritarias.
f1_macro_val = f1_score(
    y_val_sample,
    predicciones_val,
    average="macro",
    zero_division=0
)

# Comparamos las accuracy de entrenamiento y validación.
diferencia_accuracy = accuracy_train - accuracy_val

# ------------------------------------------------------------
# 5. Mostrar los resultados
# ------------------------------------------------------------

print("Modelo:", type(modelo_baseline).__name__)

print("Imágenes de validación:", len(X_val_sample))

print(
    "Accuracy entrenamiento:",
    round(accuracy_train * 100, 2),
    "%"
)

print(
    "Accuracy validación:",
    round(accuracy_val * 100, 2),
    "%"
)

print(
    "Macro-F1 validación:",
    round(f1_macro_val, 4)
)

print(
    "Diferencia de accuracy:",
    round(diferencia_accuracy * 100, 2),
    "puntos porcentuales"
)

print(
    "Tiempo de inferencia:",
    round(tiempo_inferencia, 3),
    "segundos"
)



## 10. Análisis de errores del modelo baseline

### Objetivo

Identificar qué categorías de EuroSAT clasifica
correctamente el modelo y cuáles confunde.

### Conceptos

**Matriz de confusión:**
Tabla que compara las etiquetas reales con las
predicciones realizadas por el modelo.

- Filas: categorías reales.
- Columnas: categorías predichas.
- Diagonal: predicciones correctas.
- Fuera de la diagonal: errores de clasificación.

**Precision:**
De todas las imágenes predichas como una categoría,
qué proporción pertenece realmente a ella.

**Recall:**
De todas las imágenes reales de una categoría,
qué proporción detecta correctamente el modelo.

**F1-score:**
Media armónica de precision y recall.

### Decisión técnica

Analizaremos el modelo utilizando únicamente la muestra
de validación. El conjunto de prueba permanecerá
reservado para la evaluación final.

No modificaremos el modelo hasta comprender sus errores.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 10: Matriz de confusión y análisis de errores
# ============================================================

import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Generar la matriz de confusión
# ------------------------------------------------------------

# Comparamos las categorías reales con las predichas.
#
# Indicamos explícitamente las etiquetas 0 a 9 para
# mantener el orden original de dataset.classes.
matriz_confusion = confusion_matrix(
    y_val_sample,
    predicciones_val,
    labels=range(len(dataset.classes))
)

# ------------------------------------------------------------
# 2. Visualizar la matriz
# ------------------------------------------------------------

figura, eje = plt.subplots(figsize=(12, 10))

# Mostramos los recuentos de imágenes.
# Una celda [i, j] indica cuántas imágenes reales
# de la clase i fueron clasificadas como clase j.
visualizacion = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusion,
    display_labels=dataset.classes
)

visualizacion.plot(
    ax=eje,
    cmap="Blues",
    xticks_rotation=45,
    values_format="d",
    colorbar=False
)

eje.set_title(
    "EuroSAT — Matriz de confusión del baseline",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3. Informe de clasificación
# ------------------------------------------------------------

# Este informe muestra precision, recall y F1-score
# de cada una de las diez clases.
#
# zero_division=0 evita errores cuando una clase no
# recibe ninguna predicción.
informe = classification_report(
    y_val_sample,
    predicciones_val,
    labels=range(len(dataset.classes)),
    target_names=dataset.classes,
    zero_division=0
)

print("INFORME DE CLASIFICACIÓN")
print(informe)

# ------------------------------------------------------------
# 4. Contar cuántos errores hemos cometido
# ------------------------------------------------------------

errores = (y_val_sample != predicciones_val).sum()

print("Imágenes evaluadas:", len(y_val_sample))
print("Predicciones incorrectas:", errores)
print("Predicciones correctas:", len(y_val_sample) - errores)



## 11. Experimento de mejora: escalado y regularización

### Problema detectado

El primer modelo presenta:

- Accuracy entrenamiento: 66,1 %
- Accuracy validación: 28,1 %
- Macro-F1 validación: 0,2615

También muestra diferencias importantes entre categorías.

### Hipótesis

El escalado de características y la regularización
podrían mejorar la estabilidad y generalización
del clasificador lineal.

### Experimento

Entrenaremos modelos SGDClassifier con distintos
valores de alpha, utilizando StandardScaler.

### Conceptos

**StandardScaler:**
Transforma las características para que tengan
aproximadamente media 0 y desviación estándar 1
en el conjunto de entrenamiento.

**Regularización:**
Técnica que penaliza modelos excesivamente complejos,
reduciendo el riesgo de sobreajuste.

**Pipeline:**
Estructura que encadena transformaciones y modelos.

### Prevención de data leakage

El escalador se ajustará exclusivamente utilizando
los datos de entrenamiento.

En validación se aplicará la transformación aprendida,
sin recalcular medias ni desviaciones.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 11: Experimento de escalado y regularización
# ============================================================

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Definir los valores de regularización
# ------------------------------------------------------------

# Probaremos tres valores de alpha.
#
# Un alpha mayor implica regularización más intensa.
# El mejor valor se determinará mediante validación.
valores_alpha = [0.0001, 0.001, 0.01]

# Lista donde almacenaremos los resultados.
resultados = []

# ------------------------------------------------------------
# 2. Entrenar y evaluar cada configuración
# ------------------------------------------------------------

for alpha in valores_alpha:

    print(f"Entrenando modelo con alpha={alpha}...")

    # Creamos un pipeline con dos pasos:
    #
    # 1. StandardScaler: aprende media y desviación
    #    estándar exclusivamente a partir del train.
    #
    # 2. SGDClassifier: entrena un clasificador lineal.
    #
    # Pipeline garantiza que el mismo escalado
    # se aplique durante entrenamiento y predicción.
    modelo = Pipeline([
        (
            "escalador",
            StandardScaler()
        ),
        (
            "clasificador",
            SGDClassifier(
                loss="log_loss",
                alpha=alpha,
                max_iter=1000,
                tol=1e-3,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    # Entrenamos exclusivamente con las 2.000
    # imágenes seleccionadas anteriormente.
    modelo.fit(
        X_train_sample,
        y_train_sample
    )

    # Obtenemos predicciones en validación.
    predicciones = modelo.predict(X_val_sample)

    # Calculamos accuracy.
    accuracy = accuracy_score(
        y_val_sample,
        predicciones
    )

    # Calculamos macro-F1 para dar igual peso
    # a las diez categorías.
    macro_f1 = f1_score(
        y_val_sample,
        predicciones,
        average="macro",
        zero_division=0
    )

    # Guardamos los resultados.
    resultados.append({
        "alpha": alpha,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

    print(
        f"Accuracy: {accuracy:.4f} | "
        f"Macro-F1: {macro_f1:.4f}"
    )

# ------------------------------------------------------------
# 3. Comparar las configuraciones
# ------------------------------------------------------------

tabla_resultados = pd.DataFrame(resultados)

# Ordenamos por macro-F1, que será nuestra
# métrica principal de selección.
tabla_resultados = tabla_resultados.sort_values(
    by="macro_f1",
    ascending=False
)

display(tabla_resultados)

# ------------------------------------------------------------
# 4. Identificar la mejor configuración
# ------------------------------------------------------------

mejor_resultado = tabla_resultados.iloc[0]

print(
    "Mejor alpha:",
    mejor_resultado["alpha"]
)

print(
    "Mejor Macro-F1:",
    round(mejor_resultado["macro_f1"], 4)
)



## 12. Evaluación de generalización del modelo mejorado

### Objetivo

Comparar el rendimiento en entrenamiento y validación
del modelo seleccionado mediante macro-F1.

### Resultados anteriores

Baseline:
- Accuracy entrenamiento: 66,1 %
- Accuracy validación: 28,1 %
- Macro-F1 validación: 0,2615

Mejor configuración del experimento:
- StandardScaler + SGDClassifier
- alpha = 0.001
- Accuracy validación: 41,0 %
- Macro-F1 validación: 0,3921

### Conceptos

**Generalización:** capacidad del modelo para predecir
correctamente datos no utilizados durante el entrenamiento.

**Brecha de generalización:** diferencia entre el
rendimiento en entrenamiento y validación.

**Regularización:** mecanismo que penaliza determinados
valores de los parámetros del modelo para limitar
su complejidad.

### Precaución

Una brecha pequeña no demuestra por sí sola que el
modelo sea bueno. Un modelo que obtiene resultados
bajos en ambos conjuntos también puede estar
infraajustado (underfitting).


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 12: Evaluación de generalización
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Recuperar la configuración seleccionada
# ------------------------------------------------------------

# En el experimento anterior elegimos alpha=0.001
# porque obtuvo el mayor macro-F1 en validación.
#
# Reconstruimos exactamente ese pipeline.
# No cambiamos parámetros ni muestras de datos.

modelo_mejorado = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "clasificador",
        SGDClassifier(
            loss="log_loss",
            alpha=0.001,
            max_iter=1000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        )
    )
])

# ------------------------------------------------------------
# 2. Entrenar sobre la misma muestra de 2.000 imágenes
# ------------------------------------------------------------

# fit() ajusta tanto el escalador como el clasificador
# utilizando exclusivamente los datos de entrenamiento.
modelo_mejorado.fit(
    X_train_sample,
    y_train_sample
)

# ------------------------------------------------------------
# 3. Predecir entrenamiento y validación
# ------------------------------------------------------------

pred_train_mejorado = modelo_mejorado.predict(
    X_train_sample
)

pred_val_mejorado = modelo_mejorado.predict(
    X_val_sample
)

# ------------------------------------------------------------
# 4. Calcular métricas comparables
# ------------------------------------------------------------

accuracy_train_mejorado = accuracy_score(
    y_train_sample,
    pred_train_mejorado
)

accuracy_val_mejorado = accuracy_score(
    y_val_sample,
    pred_val_mejorado
)

f1_train_mejorado = f1_score(
    y_train_sample,
    pred_train_mejorado,
    average="macro",
    zero_division=0
)

f1_val_mejorado = f1_score(
    y_val_sample,
    pred_val_mejorado,
    average="macro",
    zero_division=0
)

# ------------------------------------------------------------
# 5. Calcular las diferencias
# ------------------------------------------------------------

brecha_accuracy = (
    accuracy_train_mejorado - accuracy_val_mejorado
)

brecha_f1 = (
    f1_train_mejorado - f1_val_mejorado
)

# ------------------------------------------------------------
# 6. Mostrar resultados
# ------------------------------------------------------------

print("MODELO MEJORADO — GENERALIZACIÓN")
print("-" * 42)

print(f"Accuracy entrenamiento: {accuracy_train_mejorado:.2%}")
print(f"Accuracy validación:    {accuracy_val_mejorado:.2%}")
print(f"Brecha accuracy:        {brecha_accuracy:.2%}")

print()

print(f"Macro-F1 entrenamiento: {f1_train_mejorado:.4f}")
print(f"Macro-F1 validación:    {f1_val_mejorado:.4f}")
print(f"Brecha macro-F1:        {brecha_f1:.4f}")



## 13. Experimento con características HOG

### Objetivo

Explorar una representación alternativa de las imágenes
mediante Histogram of Oriented Gradients (HOG).

### ¿Qué es HOG?

Es un descriptor que representa la distribución de
orientaciones de gradientes (cambios de intensidad)
en distintas regiones de una imagen.

Permite describir determinados patrones de bordes
y estructuras locales.

### Hipótesis

Una representación basada en estructuras visuales
podría generalizar mejor que los píxeles RGB aplanados.

### Limitaciones

HOG puede perder información de color importante
para diferenciar categorías de cobertura del suelo.

Evaluaremos esta hipótesis mediante métricas,
sin asumir de antemano una mejora.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 13: Comprobar disponibilidad de scikit-image
# ============================================================

# importlib.metadata permite consultar la versión
# instalada de una biblioteca de Python.
from importlib.metadata import version

# hog es la función que utilizaremos para extraer
# características de bordes y gradientes.
from skimage.feature import hog

# Comprobamos que la importación funciona.
print("scikit-image:", version("scikit-image"))

print("Función HOG importada correctamente.")



## 15. Preparación de características HOG

### Objetivo

Extraer características HOG de las mismas imágenes
utilizadas en nuestro experimento anterior.

Conjuntos:
- Entrenamiento: 2.000 imágenes.
- Validación: 1.000 imágenes.
- Prueba: permanece reservada.

### Hipótesis

Una representación basada en patrones locales
de gradientes podría mejorar la generalización
respecto a los píxeles RGB aplanados.

### Decisiones técnicas

1. Conservamos los mismos índices de imágenes.
2. Aplicamos idénticos parámetros HOG a ambos conjuntos.
3. No utilizamos datos de prueba.
4. No entrenamos ningún modelo todavía.

### Limitaciones

La conversión a escala de grises elimina información
de color, que podría ser relevante para distinguir
tipos de vegetación y masas de agua.

Una reducción en el número de características no
garantiza una mejora del rendimiento.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 15: Preparación de características HOG
# ============================================================

import time
import numpy as np

from skimage.color import rgb2gray
from skimage.feature import hog

# ------------------------------------------------------------
# 1. Definir una función reutilizable
# ------------------------------------------------------------

def extraer_hog(indice):
    """
    Extrae características HOG de una imagen EuroSAT.

    Parámetros
    ----------
    indice : int
        Posición de la imagen dentro del dataset.

    Devuelve
    --------
    caracteristicas : np.ndarray
        Vector HOG de 1.764 características.

    etiqueta : int
        Categoría numérica de la imagen.
    """

    # Recuperamos la imagen y su etiqueta.
    imagen, etiqueta = dataset[int(indice)]

    # Convertimos PIL a un array RGB.
    imagen_rgb = np.asarray(imagen)

    # Convertimos a escala de grises.
    # Así aplicamos la misma configuración HOG
    # que utilizamos en el paso anterior.
    imagen_gris = rgb2gray(imagen_rgb)

    # Extraemos el descriptor HOG.
    # visualize=False evita calcular una imagen
    # adicional que no necesitamos para entrenar.
    caracteristicas = hog(
        imagen_gris,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,
        feature_vector=True,
        transform_sqrt=False
    )

    # Convertimos a float32 para reducir memoria.
    caracteristicas = caracteristicas.astype(np.float32)

    return caracteristicas, etiqueta


# ------------------------------------------------------------
# 2. Función para procesar múltiples imágenes
# ------------------------------------------------------------

def preparar_conjunto_hog(indices_conjunto):
    """
    Convierte una colección de imágenes en una matriz HOG.

    Devuelve
    --------
    X : matriz de características
    y : vector de etiquetas
    """

    vectores = []
    etiquetas_conjunto = []

    for indice in indices_conjunto:

        vector, etiqueta = extraer_hog(indice)

        vectores.append(vector)
        etiquetas_conjunto.append(etiqueta)

    # Cada fila representa una imagen.
    X = np.stack(vectores)

    # Cada elemento representa su clase real.
    y = np.array(etiquetas_conjunto)

    return X, y


# ------------------------------------------------------------
# 3. Procesar entrenamiento y validación
# ------------------------------------------------------------

inicio = time.time()

X_train_hog, y_train_hog = preparar_conjunto_hog(
    indices_muestra
)

X_val_hog, y_val_hog = preparar_conjunto_hog(
    indices_val_muestra
)

tiempo_extraccion = time.time() - inicio


# ------------------------------------------------------------
# 4. Comprobar resultados y correspondencia
# ------------------------------------------------------------

print("Forma X_train_hog:", X_train_hog.shape)
print("Forma y_train_hog:", y_train_hog.shape)

print("Forma X_val_hog:", X_val_hog.shape)
print("Forma y_val_hog:", y_val_hog.shape)

# Confirmamos que las etiquetas corresponden
# exactamente a las de los experimentos anteriores.
assert np.array_equal(y_train_hog, y_train_sample)
assert np.array_equal(y_val_hog, y_val_sample)

# Comprobamos el número de características.
assert X_train_hog.shape[1] == 1764
assert X_val_hog.shape[1] == 1764

# Verificamos que no haya valores NaN o infinitos.
assert np.isfinite(X_train_hog).all()
assert np.isfinite(X_val_hog).all()

print(
    "Tiempo de extracción:",
    round(tiempo_extraccion, 2),
    "segundos"
)

print("Matrices HOG preparadas correctamente.")



## 16. Comparación de modelos: RGB frente a HOG

### Objetivo

Comprobar si las características HOG permiten mejorar
la clasificación de imágenes EuroSAT respecto a los
valores RGB aplanados.

### Hipótesis

HOG podría ofrecer una mejor generalización al
representar patrones locales de bordes y texturas.

### Diseño experimental

Mantendremos constantes:

- Las 2.000 imágenes de entrenamiento.
- Las 1.000 imágenes de validación.
- El algoritmo SGDClassifier.
- Los parámetros de regularización y optimización.
- La métrica principal: macro-F1.

Cambiaremos:

- La representación de entrada: RGB frente a HOG.

### Limitaciones

Aunque mantengamos alpha=0.001, las características
tienen distinta dimensionalidad y distribución.
Por tanto, este experimento compara dos pipelines
concretos, no demuestra que una representación sea
universalmente superior.

El conjunto de prueba permanece reservado.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 16: Entrenamiento y evaluación con HOG
# ============================================================

import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Crear el pipeline del modelo HOG
# ------------------------------------------------------------

# Pipeline conecta dos operaciones:
#
# StandardScaler:
#   Aprende la media y desviación estándar de cada
#   característica utilizando solo entrenamiento.
#
# SGDClassifier:
#   Aprende a clasificar imágenes mediante un
#   modelo lineal con pérdida logística.
#
# Mantenemos los parámetros del modelo RGB mejorado
# para comparar ambas representaciones.

modelo_hog = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "clasificador",
        SGDClassifier(
            loss="log_loss",
            alpha=0.001,
            max_iter=1000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        )
    )
])

# ------------------------------------------------------------
# 2. Entrenar el modelo
# ------------------------------------------------------------

# Medimos únicamente el tiempo de ajuste del pipeline.
# No incluimos la extracción previa de HOG.
inicio_entrenamiento = time.perf_counter()

modelo_hog.fit(
    X_train_hog,
    y_train_hog
)

tiempo_entrenamiento_hog = (
    time.perf_counter() - inicio_entrenamiento
)

# ------------------------------------------------------------
# 3. Obtener predicciones
# ------------------------------------------------------------

# Evaluamos sobre entrenamiento para comprobar
# cuánto se ajusta el modelo a los datos conocidos.
pred_train_hog = modelo_hog.predict(X_train_hog)

# Evaluamos sobre imágenes no utilizadas
# durante el entrenamiento.
pred_val_hog = modelo_hog.predict(X_val_hog)

# ------------------------------------------------------------
# 4. Calcular las métricas
# ------------------------------------------------------------

# Accuracy: proporción de predicciones correctas.
accuracy_train_hog = accuracy_score(
    y_train_hog,
    pred_train_hog
)

accuracy_val_hog = accuracy_score(
    y_val_hog,
    pred_val_hog
)

# Macro-F1: promedio del F1 de todas las clases.
# Cada categoría tiene el mismo peso.
f1_train_hog = f1_score(
    y_train_hog,
    pred_train_hog,
    average="macro",
    zero_division=0
)

f1_val_hog = f1_score(
    y_val_hog,
    pred_val_hog,
    average="macro",
    zero_division=0
)

# ------------------------------------------------------------
# 5. Comparar RGB y HOG
# ------------------------------------------------------------

# Las variables del modelo RGB proceden del paso 14.
# Ambas evaluaciones utilizan las mismas imágenes.
comparativa = pd.DataFrame([
    {
        "Representación": "RGB",
        "Características": X_train_sample.shape[1],
        "Accuracy train": accuracy_train_mejorado,
        "Accuracy val": accuracy_val_mejorado,
        "Macro-F1 train": f1_train_mejorado,
        "Macro-F1 val": f1_val_mejorado
    },
    {
        "Representación": "HOG",
        "Características": X_train_hog.shape[1],
        "Accuracy train": accuracy_train_hog,
        "Accuracy val": accuracy_val_hog,
        "Macro-F1 train": f1_train_hog,
        "Macro-F1 val": f1_val_hog
    }
])

# Formateamos la tabla para mostrar cuatro decimales.
display(comparativa.round(4))

# ------------------------------------------------------------
# 6. Mostrar tiempo y diferencia de rendimiento
# ------------------------------------------------------------

print(
    "Tiempo de entrenamiento HOG:",
    round(tiempo_entrenamiento_hog, 2),
    "segundos"
)

# Diferencia positiva: HOG supera a RGB.
# Diferencia negativa: HOG rinde peor que RGB.
diferencia_f1 = f1_val_hog - f1_val_mejorado

print(
    "Diferencia de Macro-F1 (HOG - RGB):",
    round(diferencia_f1, 4)
)

print("Comparación completada.")



## 17. Optimización de la regularización del modelo HOG

### Objetivo

Investigar si una regularización más intensa puede
mejorar la generalización del clasificador HOG.

### Resultados del modelo HOG actual

- Alpha: 0.001
- Accuracy entrenamiento: 94,1 %
- Accuracy validación: 47,3 %
- Macro-F1 validación: 0,4625
- Brecha accuracy: 46,8 puntos porcentuales

### Hipótesis

El modelo podría estar ajustándose demasiado a las
particularidades de las 2.000 imágenes de entrenamiento.

Una regularización mayor podría reducir el sobreajuste.

### Conceptos

**Alpha:** parámetro que controla la intensidad
de la regularización en SGDClassifier.

**Sobreajuste:** el modelo obtiene resultados mucho
mejores en entrenamiento que en datos no vistos.

**Selección de hiperparámetros:** comparación de
configuraciones utilizando datos de validación.

### Diseño experimental

Mantendremos constantes:

- Las imágenes de entrenamiento y validación.
- Las características HOG.
- El escalado StandardScaler.
- El algoritmo SGDClassifier.
- La semilla aleatoria.

Modificaremos exclusivamente el valor de alpha.

El conjunto de prueba permanecerá reservado.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 17: Regularización del clasificador HOG
# ============================================================

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Definir los valores de regularización
# ------------------------------------------------------------

# Ya hemos probado alpha=0.001.
# Ahora evaluaremos valores mayores.
#
# Un alpha mayor implica una penalización más intensa
# sobre los coeficientes del clasificador.
#
# No sabemos de antemano qué valor funcionará mejor.
valores_alpha_hog = [
    0.001,
    0.01,
    0.1,
    1.0
]

# Guardaremos las métricas de cada experimento.
resultados_hog = []

# ------------------------------------------------------------
# 2. Entrenar y evaluar cada configuración
# ------------------------------------------------------------

for alpha in valores_alpha_hog:

    # Construimos un pipeline nuevo para cada alpha.
    # Así evitamos reutilizar parámetros aprendidos.
    modelo = Pipeline([
        (
            "escalador",
            StandardScaler()
        ),
        (
            "clasificador",
            SGDClassifier(
                loss="log_loss",
                alpha=alpha,
                max_iter=1000,
                tol=1e-3,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    # Entrenamos solo con la muestra de entrenamiento.
    modelo.fit(X_train_hog, y_train_hog)

    # Predecimos sobre train y validation.
    pred_train = modelo.predict(X_train_hog)
    pred_val = modelo.predict(X_val_hog)

    # Calculamos accuracy.
    acc_train = accuracy_score(y_train_hog, pred_train)
    acc_val = accuracy_score(y_val_hog, pred_val)

    # Calculamos macro-F1 para las diez categorías.
    f1_train = f1_score(
        y_train_hog,
        pred_train,
        average="macro",
        zero_division=0
    )

    f1_val = f1_score(
        y_val_hog,
        pred_val,
        average="macro",
        zero_division=0
    )

    # Guardamos todas las métricas relevantes.
    resultados_hog.append({
        "alpha": alpha,
        "accuracy_train": acc_train,
        "accuracy_val": acc_val,
        "macro_f1_train": f1_train,
        "macro_f1_val": f1_val,
        "brecha_accuracy": acc_train - acc_val
    })

    print(
        f"alpha={alpha} | "
        f"accuracy_val={acc_val:.4f} | "
        f"macro_f1_val={f1_val:.4f}"
    )

# ------------------------------------------------------------
# 3. Construir la tabla comparativa
# ------------------------------------------------------------

tabla_hog = pd.DataFrame(resultados_hog)

# Seleccionamos la configuración con mejor macro-F1
# entre los valores probados.
tabla_hog = tabla_hog.sort_values(
    by="macro_f1_val",
    ascending=False
)

display(tabla_hog.round(4))

# ------------------------------------------------------------
# 4. Identificar la mejor configuración
# ------------------------------------------------------------

mejor_hog = tabla_hog.iloc[0]

print("Mejor alpha HOG:", mejor_hog["alpha"])

print(
    "Mejor macro-F1 validación:",
    round(mejor_hog["macro_f1_val"], 4)
)



## 18. Registro y comparación de experimentos

### Objetivo

Consolidar los resultados de los modelos evaluados
durante el desarrollo del proyecto.

### Modelos analizados

1. Baseline RGB + SGDClassifier.
2. RGB + StandardScaler + SGDClassifier.
3. HOG + StandardScaler + SGDClassifier.
4. HOG con diferentes valores de regularización.

### Métrica principal

Macro-F1 de validación.

Esta métrica asigna la misma importancia a las
diez categorías y permite comparar el rendimiento
equilibrado entre clases.

### Interpretación

La selección de modelos se realiza sobre un conjunto
de validación de 1.000 imágenes.

Estas comparaciones son exploratorias: la reutilización
de la validación para tomar decisiones introduce cierto
riesgo de sobreajuste durante la selección.

El conjunto de prueba continúa reservado.

### Objetivo profesional

Mantener un registro reproducible de las decisiones
experimentales para documentarlas en GitHub.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 18: Registro de experimentos
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Definir el historial de experimentos
# ------------------------------------------------------------

# Estos resultados proceden de las ejecuciones
# anteriores de nuestro notebook.
#
# Registramos solo configuraciones realmente probadas.
# No utilizamos resultados hipotéticos.

experimentos = [
    {
        "modelo": "RGB + SGD baseline",
        "representacion": "RGB",
        "alpha": 0.0001,
        "escalado": False,
        "accuracy_val": 0.281,
        "macro_f1_val": 0.2615
    },
    {
        "modelo": "RGB + SGD optimizado",
        "representacion": "RGB",
        "alpha": 0.001,
        "escalado": True,
        "accuracy_val": 0.410,
        "macro_f1_val": 0.3921
    },
    {
        "modelo": "HOG + SGD",
        "representacion": "HOG",
        "alpha": 0.001,
        "escalado": True,
        "accuracy_val": 0.473,
        "macro_f1_val": 0.4625
    },
    {
        "modelo": "HOG + SGD",
        "representacion": "HOG",
        "alpha": 0.01,
        "escalado": True,
        "accuracy_val": 0.490,
        "macro_f1_val": 0.4732
    },
    {
        "modelo": "HOG + SGD",
        "representacion": "HOG",
        "alpha": 0.1,
        "escalado": True,
        "accuracy_val": 0.503,
        "macro_f1_val": 0.4710
    },
    {
        "modelo": "HOG + SGD",
        "representacion": "HOG",
        "alpha": 1.0,
        "escalado": True,
        "accuracy_val": 0.475,
        "macro_f1_val": 0.4024
    }
]

# ------------------------------------------------------------
# 2. Crear tabla y ordenar por macro-F1
# ------------------------------------------------------------

registro_experimentos = pd.DataFrame(experimentos)

registro_ordenado = registro_experimentos.sort_values(
    by="macro_f1_val",
    ascending=False
)

display(registro_ordenado)

# ------------------------------------------------------------
# 3. Guardar resultados en CSV
# ------------------------------------------------------------

# Creamos una carpeta para guardar resultados.
# exist_ok=True evita errores si ya existe.
carpeta_resultados = Path("results")
carpeta_resultados.mkdir(exist_ok=True)

# Guardamos los resultados sin la columna de índice.
ruta_csv = carpeta_resultados / "experiments.csv"

registro_experimentos.to_csv(
    ruta_csv,
    index=False
)

print("Archivo CSV generado:", ruta_csv)

# ------------------------------------------------------------
# 4. Comprobar que el archivo existe
# ------------------------------------------------------------

assert ruta_csv.exists()

print("Registro guardado correctamente.")


**Lectura crítica:** el mejor macro-F1 de validación observado fue aproximadamente `0.4732` (HOG, `alpha=0.01`). Es exploratorio, calculado con 1.000 imágenes de validación, y no representa rendimiento final. No se ha abierto test. El CSV escrito en `results/` es temporal en Colab: para persistirlo hay que descargarlo/subirlo a GitHub.